# 메트릭러닝 / 이미지 유사도 (Siamese) — Digit Recognizer (PyTorch) — Colab

**대조 손실(contrastive loss)** 로 임베딩을 학습해, 이미지 간 **유사도/매칭**을 수행하는 기본 예제입니다.

- 핵심 흐름: 같은 클래스 쌍은 가깝게, 다른 클래스 쌍은 멀게 **임베딩 공간**을 학습 → 최근접 검색(kNN).
- IOAI 2025 CV 과제(Restroom **Icon Matching**, 이미지 유사도)의 기반 기법입니다. (우리 11번 CLIP은 *제로샷*, 이건 *학습형* 메트릭러닝)
- [Digit Recognizer](https://www.kaggle.com/c/digit-recognizer) 데이터를 사용하며 토큰이 **노트북에 내장**되어 바로 실행됩니다.

> ⚙️ **GPU 권장**. ⚠️ 노트북에 개인 API 토큰이 평문으로 들어 있습니다. 외부 공유 금지.

## 0. Setup — 라이브러리 설치 & Kaggle 자격증명

In [ ]:
import sys, subprocess
for pkg in ["kaggle", "torch", "numpy", "pandas", "scikit-learn", "matplotlib"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
print("라이브러리 준비 완료")

In [ ]:
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"
print("Kaggle 자격증명 설정 완료 (내장 토큰)")

## 1. Kaggle에서 Digit Recognizer 데이터 다운로드

In [ ]:
import os, glob, zipfile
WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
api.competition_download_files("digit-recognizer", path=WORK_DIR, quiet=False)
for zp in glob.glob(os.path.join(WORK_DIR, "*.zip")):
    with zipfile.ZipFile(zp) as zf: zf.extractall(WORK_DIR)
    os.remove(zp)
print("다운로드 완료:", [f for f in sorted(os.listdir(WORK_DIR)) if f.endswith(".csv")])

## 2. 데이터 준비 & 쌍(pair) 구성

같은 숫자 쌍(label=1)·다른 숫자 쌍(label=0)을 무작위로 만들어 대조 학습에 사용합니다.

In [ ]:
import numpy as np, pandas as pd, torch
from torch.utils.data import Dataset, DataLoader

df = pd.read_csv(os.path.join(WORK_DIR, "train.csv"))
X = df.drop("label", axis=1).values.reshape(-1, 1, 28, 28).astype("float32") / 255.0
y = df["label"].values
# train/test 분할 (검색 평가용)
rng = np.random.RandomState(42); idx = rng.permutation(len(X)); cut = int(len(X)*0.9)
Xtr, ytr = X[idx[:cut]], y[idx[:cut]]
Xte, yte = X[idx[cut:]], y[idx[cut:]]

by_label = {c: np.where(ytr == c)[0] for c in range(10)}

class PairDataset(Dataset):
    def __init__(self, X, y, n_pairs=20000):
        self.X = X; self.y = y; self.n = n_pairs; self.rng = np.random.RandomState(0)
    def __len__(self): return self.n
    def __getitem__(self, i):
        a = self.rng.randint(len(self.X)); ca = self.y[a]
        same = self.rng.rand() < 0.5
        if same:
            b = self.rng.choice(by_label[ca])
        else:
            cb = self.rng.choice([c for c in range(10) if c != ca]); b = self.rng.choice(by_label[cb])
        return (torch.from_numpy(self.X[a]), torch.from_numpy(self.X[b]),
                torch.tensor(1.0 if same else 0.0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loader = DataLoader(PairDataset(Xtr, ytr), batch_size=128, shuffle=True)
print("device:", device, "| train:", len(Xtr), "test:", len(Xte))

## 3. 임베딩 네트워크 & 대조 손실

In [ ]:
import torch.nn as nn, torch.nn.functional as F

class Embedder(nn.Module):
    def __init__(self, dim=64):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 14
            nn.Conv2d(32,64,3,padding=1), nn.ReLU(), nn.MaxPool2d(2), # 7
        )
        self.fc = nn.Linear(64*7*7, dim)
    def forward(self, x):
        z = self.fc(self.conv(x).flatten(1))
        return F.normalize(z, dim=1)  # 단위 구면 임베딩

def contrastive_loss(za, zb, same, margin=1.0, eps=1e-9):
    d2 = (za - zb).pow(2).sum(1)         # 제곱 유클리드 거리
    d = torch.sqrt(d2 + eps)             # 유클리드 거리 (eps 로 sqrt(0) 그래디언트 안정화)
    # 같은 클래스: 가깝게(d2), 다른 클래스: margin 밖으로 밀어냄
    return (same * d2 + (1 - same) * F.relu(margin - d).pow(2)).mean()

model = Embedder().to(device)
print("Embedder 준비 완료")

## 4. 학습

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 5
for epoch in range(1, EPOCHS + 1):
    model.train(); running = 0.0; n = 0
    for a, b, same in loader:
        a, b, same = a.to(device), b.to(device), same.to(device)
        loss = contrastive_loss(model(a), model(b), same)
        opt.zero_grad(); loss.backward(); opt.step()
        running += loss.item() * a.size(0); n += a.size(0)
    print(f"epoch {epoch} | contrastive loss = {running/n:.4f}")

## 5. 임베딩 분리도 평가 & kNN 검색

같은 클래스 거리 < 다른 클래스 거리인지 확인하고, kNN 으로 검색 정확도를 측정합니다.

In [ ]:
@torch.no_grad()
def embed(arr, bs=512):
    model.eval(); outs = []
    for i in range(0, len(arr), bs):
        outs.append(model(torch.from_numpy(arr[i:i+bs]).to(device)).cpu().numpy())
    return np.concatenate(outs)

Etr, Ete = embed(Xtr), embed(Xte)

# 같은/다른 클래스 평균 거리
r = np.random.RandomState(1); ia, ib = r.randint(len(Etr), size=2000), r.randint(len(Etr), size=2000)
dist = np.linalg.norm(Etr[ia] - Etr[ib], axis=1); same = ytr[ia] == ytr[ib]
print(f"same-class 평균거리 {dist[same].mean():.3f} < diff-class 평균거리 {dist[~same].mean():.3f}")

# kNN(k=1) 검색 정확도: 테스트 임베딩의 최근접 train 라벨
from sklearn.neighbors import KNeighborsClassifier
knn = KNeighborsClassifier(n_neighbors=1).fit(Etr, ytr)
acc = (knn.predict(Ete) == yte).mean()
print(f"kNN(k=1) 검색 정확도: {acc:.4f}")

## 6. 검색 결과 시각화

쿼리 이미지와 임베딩 공간에서 가장 가까운 이웃들을 나란히 보여줍니다.

In [ ]:
import matplotlib.pyplot as plt
q = 0
dists = np.linalg.norm(Etr - Ete[q], axis=1)
nn_idx = dists.argsort()[:5]
fig, ax = plt.subplots(1, 6, figsize=(12, 2.5))
ax[0].imshow(Xte[q,0], cmap="gray"); ax[0].set_title(f"query: {yte[q]}"); ax[0].axis("off")
for k, j in enumerate(nn_idx):
    ax[k+1].imshow(Xtr[j,0], cmap="gray"); ax[k+1].set_title(f"nn{k+1}: {ytr[j]}"); ax[k+1].axis("off")
plt.show()

## 🏁 제출 안내

이 노트북은 **이미지 유사도/검색(메트릭러닝)** 데모라 제출 리더보드가 없습니다.

- 데이터 출처: **[Digit Recognizer](https://www.kaggle.com/c/digit-recognizer)**

> IOAI 2025 CV 과제(이미지 매칭/유사도)의 기반 기법입니다.